In [1]:
import os

In [2]:
os.chdir("../")

#### Update Entity

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

#### Update Configuration Manager

In [4]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, params_file_path: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        model_evaluation_config = self.config.model_evaluation

        create_directories([model_evaluation_config.root_dir])

        return ModelEvaluationConfig(
            root_dir=Path(model_evaluation_config.root_dir),
            data_path=Path(model_evaluation_config.data_path),
            model_path=Path(model_evaluation_config.model_path),
            tokenizer_path=Path(model_evaluation_config.tokenizer_path),
            metric_file_name=Path(model_evaluation_config.metric_file_name)
        )

#### Update Components

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
load_metric = evaluate.load
from datasets import load_from_disk, load_dataset
import torch
import pandas as pd
from tqdm import tqdm

[2026-02-26 12:36:55,536: INFO: config]: TensorFlow version 2.20.0 available.


In [7]:
class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self, dataset, metric, model, tokenizer,
                                    batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                    column_text="article",
                                    column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            inputs = tokenizer(article_batch, max_length=1023, truncation=True,
                                padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                attention_mask=inputs["attention_mask"].to(device),
                                length_penalty=0.8, num_beams=8, max_length=128)
            ''' parameter for length penalty ensures that the model does not generate
            sequences that are too long. '''

            # Decode the generated texts
            decoded_summaries = tokenizer.batch_decode(summaries, skip_special_tokens=True, clean_up_tokenization_spaces=True)

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        # Finally compute and return the ROUGE scores.
        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        #loading dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

        rouge_metric = load_metric("rouge")

        score = self.calculate_metric_on_test_ds(dataset_samsum_pt["test"][0:10], rouge_metric, 
                                                 model_pegasus, tokenizer, batch_size=2,
                                                 column_text="dialogue", column_summary="summary")
        
        rouge_dict = dict((rn, score[rn]) for rn in rouge_names)
        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file_name, index=False)


#### Update the Pipeline

In [8]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(model_evaluation_config)
    model_evaluation.evaluate()
except Exception as e:
    raise e

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

100%|██████████| 5/5 [05:15<00:00, 63.06s/it]

[2026-02-26 12:42:16,732: INFO: rouge_scorer]: Using default tokenizer.
